# QTS Project  
## Option Wheel Strategy

This project studies a systematic options wheel strategy implemented on listed American equity options. The framework integrates multi-factor stock selection with option premium harvesting under a margin-based capital structure.



---

**Course:** Quantitative Trading Strategy  
**Group:** Final Project Group PF: J 

**Group Members**

- **Name**: Mingshu Lu   **Student ID**: 12496646
- **Name**: Jackie Zhang   **Student ID**: 12498155
- **Name**: Theo Li          **Student ID**: 12503045
- **Name**: Jessica Xu       **Student ID**: 12503042
- **Name**: Catherine Chen   **Student ID**: 12496600

# Project Overview

This project studies a systematic options wheel strategy implemented on listed American-style equity options written on U.S. large-cap stocks. The framework combines equity selection and option premium harvesting within a unified backtesting structure. We'll use the data from 2013-04-01 to 2024-12-31. The objective is to evaluate whether disciplined short-volatility exposure, when applied to a selected set of liquid underlying equities, can generate attractive risk-adjusted returns over an extended sample period.

The strategy operates through monthly rolling option positions while maintaining a multi-year investment horizon to ensure exposure across different market regimes. Both assignment and premium collection dynamics are explicitly modeled, allowing the portfolio to alternate between cash-secured puts and covered calls. Capital allocation, transaction costs, and execution frictions are incorporated to ensure realistic performance estimation.

The backtest spans more than five years and includes at least five distinct underlying equities, generating sufficient trade frequency for statistical evaluation. Performance is assessed across full-sample and stressed market environments in order to evaluate robustness under varying volatility conditions.

# 0. Strategy Architecture

This session defines the structural architecture of the strategy and clarifies how the equity selection layer interacts with the options execution layer. The objective is to establish a clear mapping between conceptual design and the implementation that follows in code.

The strategy consists of two interacting components:

1. A semiannual equity selection mechanism
2. A monthly options wheel execution mechanism

The equity layer determines the eligible stock universe at each rebalance date, while the option layer generates recurring premium income conditional on that selected universe.

Rebalance frequencies are defined as:

$$
T_{equity} = 6 \text{ months}
$$

$$
T_{option} = 4 \text{ weeks}
$$

The code in subsequent sections will construct these two timing cycles explicitly and implement their interaction within a unified backtesting engine.

# 1. Equity Selection

This session constructs the equity selection mechanism that determines which underlying stocks are eligible for option writing. The objective is to formalize the stock ranking process and produce a time-indexed selected stock set.

The data sources are as follows:
- Daily stock market cap, fundamental, analyst consensus data from WRDS(CRSP/IBES)

The selection rule is based on a composite factor score defined as:

$$
Score_i = 0.5 MV_i + 0.2 Q_i + 0.2 M_i + 0.1 C_i
$$

where $MV_i$, $Q_i$, $M_i$, and $C_i$ denote standardized market capitalization, quality, momentum, and analyst consensus signals.

At each semiannual rebalance date, stocks are ranked cross-sectionally and the top-ranked securities are selected. The selected set is denoted as:

$$
\mathcal{S}_t = \{ i_1, i_2, ..., i_k \}
$$

The code in this session will compute factor signals, perform cross-sectional ranking, and construct the time-series of selected stock pools.

In [ ]:
import pandas as pd
import numpy as np
import wrds

# ==========================================
# 1. Data Ingestion: WRDS Pipeline
# ==========================================
def build_sp500_universe_pipeline(username: str, start_date: str = '2011-01-01', end_date: str = '2024-12-31') -> pd.DataFrame:
    """
    Fetches CRSP pricing (including shares outstanding for Market Cap), 
    Compustat fundamentals, and IBES analyst consensus.
    """
    db = wrds.Connection(wrds_username=username)
    
    # print("Fetching CRSP daily prices and calculating Market Cap...")
    crsp_query = f"""
        SELECT a.date, b.ticker, a.prc, a.shrout
        FROM crsp.dsf AS a
        JOIN crsp.dsenames AS b ON a.permno = b.permno
        JOIN crsp.msp500list AS sp ON a.permno = sp.permno
        WHERE a.date >= '{start_date}' AND a.date <= '{end_date}'
        AND a.date >= sp.start AND (a.date <= sp.ending OR sp.ending IS NULL)
        AND a.date >= b.namedt AND a.date <= b.nameendt
    """
    df_price = db.raw_sql(crsp_query, date_cols=['date'])
    df_price['prc'] = df_price['prc'].abs()
    
    # Calculate Market Cap: Price * Shares Outstanding (shrout is usually in thousands)
    df_price['shrout'] = df_price['shrout'].fillna(0)
    df_price['market_cap'] = df_price['prc'] * df_price['shrout']
    
    # Calculate daily return and 6-month momentum
    df_price = df_price.sort_values(['ticker', 'date'])
    df_price['daily_return'] = df_price.groupby('ticker')['prc'].pct_change(1)
    df_price['mom6'] = df_price.groupby('ticker')['prc'].pct_change(126) 
    
    sp500_tickers = df_price['ticker'].dropna().unique().tolist()
    ticker_str = str(tuple(sp500_tickers)) if len(sp500_tickers) > 1 else f"('{sp500_tickers[0]}')"
        
    # print("Fetching Compustat fundamentals (Quality)...")
    comp_query = f"""
        SELECT datadate AS date, tic AS ticker, ni, seq
        FROM comp.funda
        WHERE tic IN {ticker_str}
        AND datadate >= '{start_date}' AND datadate <= '{end_date}'
        AND indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C'
    """
    df_fund = db.raw_sql(comp_query, date_cols=['date'])
    df_fund['quality'] = df_fund['ni'] / df_fund['seq'].replace(0, np.nan)
    df_fund['quality'] = df_fund['quality'].astype(float)
    
    # print("Fetching IBES analyst consensus...")
    ibes_query = f"""
        SELECT statpers AS date, ticker, meanrec AS analyst_consensus
        FROM ibes.recdsum
        WHERE ticker IN {ticker_str}
        AND statpers >= '{start_date}' AND statpers <= '{end_date}'
    """
    df_ibes = db.raw_sql(ibes_query, date_cols=['date'])
    
    db.close()
    
    # print("Merging panel data...")
    df_price.set_index(['date', 'ticker'], inplace=True)
    df_fund.set_index(['date', 'ticker'], inplace=True)
    df_ibes.set_index(['date', 'ticker'], inplace=True)
    
    panel = df_price[['prc', 'daily_return', 'mom6', 'market_cap']].copy()
    panel = panel.join(df_fund[['quality']], how='left')
    panel = panel.join(df_ibes[['analyst_consensus']], how='left')
    
    panel['quality'] = panel.groupby(level='ticker')['quality'].ffill()
    panel['analyst_consensus'] = panel.groupby(level='ticker')['analyst_consensus'].ffill()
    
    panel = panel.reset_index()
    panel = panel[(panel['date'] >= '2013-04-01')]
    
    return panel.dropna()


# ==========================================
# 2. Rebalance Schedule (Semiannual)
# ==========================================
def generate_semiannual_rebalance_dates(panel: pd.DataFrame, start_date: str = '2013-04-01', end_date: str = '2024-12-31') -> list:
    """
    Generates semiannual (every 6 months) rebalance dates.
    Aligns the target dates to the next available valid trading day.
    """
    # print("Generating semiannual rebalance dates...")
    trading_dates = pd.DatetimeIndex(pd.Series(panel['date'].unique()).sort_values())
    
    # Create target dates every 6 months (e.g., Apr 1 and Oct 1)
    target_dates = pd.date_range(start=start_date, end=end_date, freq=pd.DateOffset(months=6))
    
    rebalance_dates = []
    for target in target_dates:
        # Find the first valid trading day on or after the target date
        valid_dates = trading_dates[trading_dates >= target]
        if len(valid_dates) > 0:
            rebalance_dates.append(valid_dates[0].strftime('%Y-%m-%d'))
            
    return rebalance_dates

# ==========================================
# 3. Factor Scoring & Selection
# ==========================================
def generate_universe_table(panel: pd.DataFrame, rebalance_dates: list) -> pd.DataFrame:
    """
    Applies the new 4-factor formula: Score = 0.5*MV + 0.2*Q + 0.2*M + 0.1*C
    """
    # print("Calculating composite scores and ranking equities...")
    rebal_df = panel[panel['date'].isin(pd.to_datetime(rebalance_dates))].copy()
    rebal_df.rename(columns={'date': 'rebalance_date'}, inplace=True)
    
    factors = ['market_cap', 'quality', 'mom6', 'analyst_consensus']
    
    # Step 1: Cross-sectional Z-score standardization
    for factor in factors:
        rebal_df[f'{factor}_z'] = rebal_df.groupby('rebalance_date')[factor].transform(
            lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0
        )
        
    # Step 2: Apply the weighted formula
    # NOTE: analyst_consensus is subtracted because a lower IBES score (1) means 'Strong Buy'
    rebal_df['composite_score'] = (
        0.5 * rebal_df['market_cap_z'] + 
        0.2 * rebal_df['quality_z'] + 
        0.2 * rebal_df['mom6_z'] - 
        0.1 * rebal_df['analyst_consensus_z']
    )
    
    # Step 3: Rank and Select Top 20
    rebal_df = rebal_df[rebal_df['prc'] <= 1000.0].copy()
    rebal_df['rank'] = rebal_df.groupby('rebalance_date')['composite_score'].rank(method='first', ascending=False)
    rebal_df['selected_flag'] = (rebal_df['rank'] <= 20).astype(int)
    rebal_df['weight'] = np.where(rebal_df['selected_flag'] == 1, 0.05, 0.0)
    
    # output_cols = ['rebalance_date', 'ticker', 'market_cap', 'quality', 'mom6', 'analyst_consensus', 
    #                'composite_score', 'rank', 'selected_flag', 'weight']
    
    output_cols = ['rebalance_date', 'ticker','rank', 'selected_flag']

    final_universe = rebal_df[output_cols].sort_values(['rebalance_date', 'rank'])
    final_universe = final_universe[final_universe['selected_flag'] == 1].reset_index(drop=True)
    
    return final_universe

if __name__ == "__main__":
    YOUR_USERNAME = 'your_username' 
    
    # Run pipelines
    final_panel = build_sp500_universe_pipeline(username=YOUR_USERNAME)
    semi_annual_dates = generate_semiannual_rebalance_dates(final_panel)
    
    final_universe = generate_universe_table(panel=final_panel, rebalance_dates=semi_annual_dates)

    final_universe.to_parquet('UniverseTable.parquet', index=False)

Loading library list...
Done


/var/folders/zs/t66fm21n3rndqc_bp6z244200000gn/T/ipykernel_5155/1601911464.py:34: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_price['daily_return'] = df_price.groupby('ticker')['prc'].pct_change(1)
/var/folders/zs/t66fm21n3rndqc_bp6z244200000gn/T/ipykernel_5155/1601911464.py:35: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_price['mom6'] = df_price.groupby('ticker')['prc'].pct_change(126)


# 2. Data Loading, Cleaning, and Engineering Pipeline

This section defines the data infrastructure and preprocessing pipeline used to construct the research dataset. The objective is to transform raw equity and option data into a clean, aligned panel suitable for signal construction and backtesting.

The data sources are as follows:

- Daily listed option data from Databento API
- Daily stock price, fundamental, analyst consensus data from WRDS / IBES

All datasets are aligned to a common trading calendar and indexed by date and underlying ticker.

---

## 2.1 Equity Data Loading and Cleaning

This subsection loads daily equity price data from WRDS for the selected stocks.
The output of this step is a cleaned equity panel indexed by date and ticker, which serves as the foundation for factor ranking and portfolio construction.

In [ ]:
if __name__ == "__main__":
    YOUR_USERNAME = 'your_username' 
    
    # Run pipelines
    final_panel = build_sp500_universe_pipeline(username=YOUR_USERNAME)
    
    # Generate Table 2: UnderlyingPrice
    underlying_price = final_panel[['date', 'ticker', 'prc']].rename(columns={'prc': 'close'})
    
    underlying_price.to_parquet('UnderlyingPrice.parquet', index=False)

Loading library list...
Done


/var/folders/zs/t66fm21n3rndqc_bp6z244200000gn/T/ipykernel_5155/1601911464.py:34: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_price['daily_return'] = df_price.groupby('ticker')['prc'].pct_change(1)
/var/folders/zs/t66fm21n3rndqc_bp6z244200000gn/T/ipykernel_5155/1601911464.py:35: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_price['mom6'] = df_price.groupby('ticker')['prc'].pct_change(126)




---

## 2.2 Option Data Loading and Cleaning

This subsection loads daily option data from Databento API. The raw dataset includes option price, strike, expiration date, implied volatility, option delta, and open interest.

Contracts are filtered to approximately 4-week maturity in order to match the monthly rolling design of the wheel strategy. Illiquid contracts are removed based on open interest thresholds. Option records are merged with underlying equity prices to ensure pricing consistency.

The output is an aligned option panel indexed by date, underlying ticker, and contract characteristics.



---

## 2.3 3-Month T-bill Data Loading and Cleaning

This subsection loads daily 3-month T-bill rate data.

In [17]:
import pandas_datareader.data as web

def fetch_risk_free_rate(start_date: str = '2013-04-01', end_date: str = '2024-12-31') -> pd.DataFrame:
    """
    Fetches the 3-Month Treasury Bill Secondary Market Rate (DTB3) from FRED.
    """
    # print("Fetching 3-Month T-Bill Risk-Free Rate from FRED...")
    rf = web.DataReader('DTB3', 'fred', start_date, end_date)
    rf = rf.reset_index()
    rf.rename(columns={'DATE': 'date', 'DTB3': 'r_3m_annual_pct'}, inplace=True)
    rf['r_3m_annual_pct'] = rf['r_3m_annual_pct'].ffill()
    return rf

if __name__ == "__main__":
    
    risk_free_df = fetch_risk_free_rate()
    risk_free_df.to_parquet('RiskFree3M.parquet', index=False)

# 3. Option Delta Calculation

This session formalizes the calculation of option delta, which is a critical input for strike selection in the options wheel strategy. The objective is to implement a robust method for computing equity option delta based on the Black-Scholes model and modifications for American options..

# 4. Option Wheel Mechanics

This session formalizes the mechanical implementation of the options wheel strategy. The objective is to translate the conceptual wheel process into executable trade rules.

For each selected stock, a 4-week out-of-the-money put is sold. Strike selection is determined by delta targeting, typically under 10-delta and 20-delta regimes. Assignment probability is approximated by:

$$
P(\text{assignment}) \approx |\Delta|
$$

If the put expires in-the-money, the underlying stock is assigned. The strategy then transitions to a covered call position. If the call expires in-the-money, the stock is called away and the process returns to a cash-secured put phase.

The code in this session will simulate option expiration outcomes, handle assignment logic, and implement the state transition between cash, stock holding, and covered call positions.

# 5.Strategy Execution

This session formalizes the execution logic of the strategy.

# 5.1 Capital Allocation and Margin Model

This session defines the capital base and leverage framework under which the strategy operates. The objective is to ensure that position sizing and funding costs are explicitly modeled.

Initial capital is defined as:

$$
C_0 = 10{,}000{,}000
$$

Position sizes are determined subject to margin requirements. If a Reg-T framework is applied, position exposure must satisfy:

$$
Exposure \leq \frac{Capital}{Margin\ Requirement}
$$

If assignment results in stock ownership, borrowing costs are incorporated through a funding rate $r_{borrow}$.

The code in this session will compute margin-adjusted position sizes, update available capital dynamically, and incorporate funding costs into portfolio PnL.

# 5.2 Transaction Costs and Execution Modeling

This session incorporates realistic execution assumptions into the backtest. The objective is to prevent overestimation of strategy performance.

Option commissions are modeled as:

$$
0.3 \text{ USD per contract}
$$

Slippage is modeled as:

$$
P_{exec} = P_{mid}(1 \pm 0.01)
$$

where execution price deviates by 1% from mid-price.

The code in this session will adjust trade prices for commission and slippage and produce both gross and net performance series.

# 5.3 Backtest Engine

This session integrates all previous components into a unified portfolio simulation framework. The objective is to generate a daily time series of portfolio value.

Portfolio value is defined as:

$$
V_t = C_t + \sum Equity_t + \sum Option_t
$$

where $C_t$ denotes cash balance, and positions are marked to market daily.

The code in this session will iterate through time, execute monthly option rolls, update assignment states, adjust capital, and record daily portfolio value.

# 6. Performance Evaluation

This session evaluates the performance of the strategy over the full sample and selected subperiods. The objective is to quantify return, risk, and drawdown characteristics.

Key performance measures include:

- Annualized return  
- Annualized volatility  
- Sharpe ratio  
- Maximum drawdown  
- Value-at-Risk (VaR)  

The code in this session will compute these metrics for both gross and net returns and compare them across volatility regimes.